In [13]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

In [14]:
poi_df = pd.read_csv("data/station_pois.csv")
crime_df = pd.read_csv("data/crime_within_station_walksheds.csv")
stations = gpd.GeoDataFrame(
    {
        "name": [
            "9th Street","7th Street","CTC/Arena","3rd St/Convention",
            "Brooklyn Village","Carson","Bland","East/West"
        ],
        "lat": [35.22970,35.22722,35.22500,35.22361,35.22139,35.21889,35.21583,35.21194],
        "lon": [-80.83500,-80.83806,-80.84139,-80.84306,-80.84694,-80.85083,-80.85528,-80.85917]
    },
    geometry=[Point(xy) for xy in zip([-80.83500,-80.83806,-80.84139,-80.84306,-80.84694,-80.85083,-80.85528,-80.85917],
                                      [35.22970,35.22722,35.22500,35.22361,35.22139,35.21889,35.21583,35.21194])],
    crs="EPSG:4326"
)
stations_walkshed_4326 = gpd.read_file("data/station_walksheds.geojson")

In [15]:
def assign_station_lists(points_gdf: gpd.GeoDataFrame,
                         stations_points_4326: gpd.GeoDataFrame,
                         walksheds_4326: gpd.GeoDataFrame,
                         station_name_col: str = "name",
                         out_col: str = "stations_in_radius",
                         out_dist_col: str = "stations_in_radius_dist_m",
                         crs_meters: str = "EPSG:26917") -> gpd.GeoDataFrame:
    """
    For each point, find ALL station walksheds that contain it (within),
    then return ordered station name list (closest->farthest) by distance
    to the station POINT geometry (in meters).
    """

    # Keep a stable id for grouping (index can get messy after joins)
    pts = points_gdf.copy()
    pts["_pt_id"] = range(len(pts))

    # Project to a meters CRS for distance calcs
    pts_m = pts.to_crs(crs_meters)
    walksheds_m = walksheds_4326[[station_name_col, "geometry"]].to_crs(crs_meters)
    stations_m = stations_points_4326[[station_name_col, "geometry"]].to_crs(crs_meters)

    # Spatial join: each point may match multiple walksheds
    joined = gpd.sjoin(
        pts_m[["_pt_id", "geometry"]],
        walksheds_m,
        predicate="within",
        how="left"
    ).rename(columns={station_name_col: "_station"})

    # Attach the station POINT geometry so we can compute point->station distance
    stations_m2 = stations_m.rename(columns={"geometry": "_station_geom"})
    joined = joined.merge(
        stations_m2[[station_name_col, "_station_geom"]].rename(columns={station_name_col: "_station"}),
        on="_station",
        how="left"
    )

    # Distance in meters (NaN where no station matched)
    joined["_dist_m"] = joined.geometry.distance(joined["_station_geom"])

    # Build ordered lists per point
    def agg_station_lists(df):
        df = df.dropna(subset=["_station", "_dist_m"]).sort_values("_dist_m")
        return pd.Series({
            out_col: df["_station"].tolist(),
            out_dist_col: [round(x, 2) for x in df["_dist_m"].tolist()]
        })

    agg = joined.groupby("_pt_id", as_index=False).apply(agg_station_lists).reset_index(drop=True)

    # Join results back to original points
    pts_out = pts.merge(agg, on="_pt_id", how="left")
    pts_out[out_col] = pts_out[out_col].apply(lambda x: x if isinstance(x, list) else [])
    pts_out[out_dist_col] = pts_out[out_dist_col].apply(lambda x: x if isinstance(x, list) else [])

    # Cleanup
    pts_out = pts_out.drop(columns=["_pt_id"])

    return pts_out

In [ ]:
poi_gdf = gpd.GeoDataFrame(
    poi_df,
    geometry=gpd.points_from_xy(poi_df["lon"], poi_df["lat"]),
    crs="EPSG:4326"
)

poi_with_lists = assign_station_lists(
    points_gdf=poi_gdf,
    stations_points_4326=stations,
    walksheds_4326=stations_walkshed_4326,
    station_name_col="name",
    out_col="stations_in_radius",
    out_dist_col="stations_in_radius_dist_m"
)

# --- Save as GeoJSON (lists are fine in GeoJSON) ---
poi_with_lists = poi_with_lists.drop(columns=['station_id', 'osm_type', 'osm_key'])
poi_with_lists.to_file("data/pois_with_station_lists.geojson", driver="GeoJSON")

# --- Save as CSV (lists should be stringified) ---
poi_csv = poi_with_lists.copy()
poi_csv["stations_in_radius"] = poi_csv["stations_in_radius"].apply(lambda x: ",".join(x))
poi_csv["stations_in_radius_dist_m"] = poi_csv["stations_in_radius_dist_m"].apply(lambda x: ",".join(map(str, x)))
poi_csv.drop(columns="geometry").to_csv("data/pois_with_station_lists.csv", index=False)
poi_csv.head(20)

,osm_id,name,lat,lon,tags,amenity,shop,leisure,tourism,office,public_transport,healthcare,craft,geometry,stations_in_radius,stations_in_radius_dist_m
0,357772806,First Ward Elementary School,35.228198,-80.833404,"{'amenity': 'school', 'ele': '228', 'gnis:feat...",school,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-80.8334 35.2282),"9th Street,7th Street","220.95,437.32"
1,367908585,Charlotte Fire Department Station 4,35.232018,-80.839129,"{'addr:city': 'Charlotte', 'addr:housenumber':...",fire_station,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-80.83913 35.23202),9th Street,455.21
2,957284836,6th Street,35.226627,-80.833456,"{'amenity': 'parking', 'name': '6th Street', '...",parking,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-80.83346 35.22663),"9th Street,7th Street","368.59,424.11"
3,4001585783,Aroma,35.229878,-80.839511,"{'addr:city': 'Charlotte', 'addr:housenumber':...",restaurant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-80.83951 35.22988),"7th Street,9th Street","323.04,411.0"
4,4001610982,NaN,35.229738,-80.839606,"{'amenity': 'bicycle_parking', 'bicycle_parkin...",bicycle_parking,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-80.83961 35.22974),"7th Street,9th Street","312.71,419.16"
5,4001610992,NaN,35.229504,-80.840018,{'amenity': 'waste_basket'},waste_basket,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-80.84002 35.2295),"7th Street,9th Street","309.7,457.11"
6,6532532810,7th Restaurant & Lounge,35.226134,-80.836090,"{'addr:city': 'Charlotte', 'addr:housenumber':...",restaurant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-80.83609 35.22613),"7th Street,9th Street","215.98,407.71"
7,9267613529,NaN,35.227393,-80.836323,"{'amenity': 'bench', 'armrest': 'yes', 'backre...",bench,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-80.83632 35.22739),"7th Street,9th Street","159.24,282.74"
8,9267613530,NaN,35.227585,-80.836069,"{'amenity': 'bench', 'armrest': 'yes', 'backre...",bench,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-80.83607 35.22758),"7th Street,9th Street","185.68,253.92"
9,9267613531,NaN,35.227817,-80.835974,"{'amenity': 'bench', 'armrest': 'yes', 'backre...",bench,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-80.83597 35.22782),"7th Street,9th Street","201.07,226.82"


In [ ]:
crime_gdf = gpd.GeoDataFrame(
    crime_df,
    geometry=gpd.points_from_xy(crime_df["LONGITUDE_PUBLIC"], crime_df["LATITUDE_PUBLIC"]),
    crs="EPSG:4326"
)

crime_with_lists = assign_station_lists(
    points_gdf=crime_gdf,
    stations_points_4326=stations,
    walksheds_4326=stations_walkshed_4326,
    station_name_col="name",
    out_col="stations_in_radius",
    out_dist_col="stations_in_radius_dist_m"
)

# Save GeoJSON
crime_with_lists = crime_with_lists.drop(columns=['lat', 'lon', 'index_right'])
crime_with_lists = crime_with_lists.rename(columns={'name': 'nearest_station'})
crime_with_lists.to_file("data/crimes_with_station_lists.geojson", driver="GeoJSON")

# Save CSVs (stringify list columns)
def stringify_lists_for_csv(df):
    df = df.copy()
    df["stations_in_radius"] = df["stations_in_radius"].apply(lambda x: ",".join(x))
    df["stations_in_radius_dist_m"] = df["stations_in_radius_dist_m"].apply(lambda x: ",".join(map(str, x)))
    return df.drop(columns="geometry")

stringify_lists_for_csv(crime_with_lists).to_csv("data/crimes_with_station_lists.csv", index=False)
crime_with_lists.head(20)

,LOCATION,ZIP,LATITUDE_PUBLIC,LONGITUDE_PUBLIC,CMPD_PATROL_DIVISION,NPA,DATE_INCIDENT_BEGAN,LOCATION_TYPE_DESCRIPTION,PLACE_TYPE_DESCRIPTION,PLACE_DETAIL_DESCRIPTION,HIGHEST_NIBRS_DESCRIPTION,cluster,cluster_title,geometry,index_right,name,lat,lon,stations_in_radius,stations_in_radius_dist_m
0,200 E TRADE ST,28202,35.226243,-80.841973,Central,476,2018/11/17 00:00:00+00,Outdoors,Open Area,Street/Highway,Drug/Narcotic Violations,1,"""Regulatory and Non-Violent Offenses""",POINT (-80.84197 35.22624),1,7th Street,35.22722,-80.83806,"[CTC/Arena, 3rd St/Convention, 7th Street]","[147.71, 308.3, 372.2]"
1,200 E TRADE ST,28202,35.226243,-80.841973,Central,476,2018/11/17 00:00:00+00,Outdoors,Open Area,Street/Highway,Drug/Narcotic Violations,1,"""Regulatory and Non-Violent Offenses""",POINT (-80.84197 35.22624),2,CTC/Arena,35.22500,-80.84139,"[CTC/Arena, 3rd St/Convention, 7th Street]","[147.71, 308.3, 372.2]"
2,200 E TRADE ST,28202,35.226243,-80.841973,Central,476,2018/11/17 00:00:00+00,Outdoors,Open Area,Street/Highway,Drug/Narcotic Violations,1,"""Regulatory and Non-Violent Offenses""",POINT (-80.84197 35.22624),3,3rd St/Convention,35.22361,-80.84306,"[CTC/Arena, 3rd St/Convention, 7th Street]","[147.71, 308.3, 372.2]"
3,400 S DAVIDSON ST,28204,35.220123,-80.841999,Central,476,2018/12/21 00:00:00+00,Outdoors,Open Area,Street/Highway,Theft From Motor Vehicle,8,Theft and Vehicle Offenses,POINT (-80.842 35.22012),3,3rd St/Convention,35.22361,-80.84306,"[3rd St/Convention, Brooklyn Village]","[398.58, 471.1]"
4,400 S DAVIDSON ST,28204,35.220123,-80.841999,Central,476,2018/12/21 00:00:00+00,Outdoors,Open Area,Street/Highway,Theft From Motor Vehicle,8,Theft and Vehicle Offenses,POINT (-80.842 35.22012),4,Brooklyn Village,35.22139,-80.84694,"[3rd St/Convention, Brooklyn Village]","[398.58, 471.1]"
5,600 E TRADE ST,28202,35.222915,-80.838020,Central,476,2020/04/10 00:00:00+00,Other,Open Area,Cyberspace,Pornography/Obscene Material,5,Sexual Exploitation Crimes,POINT (-80.83802 35.22292),1,7th Street,35.22722,-80.83806,"[CTC/Arena, 7th Street]","[384.08, 477.44]"
6,600 E TRADE ST,28202,35.222915,-80.838020,Central,476,2020/04/10 00:00:00+00,Other,Open Area,Cyberspace,Pornography/Obscene Material,5,Sexual Exploitation Crimes,POINT (-80.83802 35.22292),2,CTC/Arena,35.22500,-80.84139,"[CTC/Arena, 7th Street]","[384.08, 477.44]"
7,400 N TRYON ST,28202,35.229505,-80.839786,Central,476,2017/06/25 00:00:00+00,Outdoors,Commercial Place,Restaurant/Diner/Coffee Shop,Purse-Snatching,4,"""Theft-Related Offenses""",POINT (-80.83979 35.2295),0,9th Street,35.22970,-80.83500,"[7th Street, 9th Street]","[298.13, 436.04]"
8,400 N TRYON ST,28202,35.229505,-80.839786,Central,476,2017/06/25 00:00:00+00,Outdoors,Commercial Place,Restaurant/Diner/Coffee Shop,Purse-Snatching,4,"""Theft-Related Offenses""",POINT (-80.83979 35.2295),1,7th Street,35.22722,-80.83806,"[7th Street, 9th Street]","[298.13, 436.04]"
9,200 EAST BV,28203,35.211751,-80.857709,Central,3,2021/05/24 00:00:00+00,Indoors,Retail,Convenience Store,Shoplifting,4,"""Theft-Related Offenses""",POINT (-80.85771 35.21175),7,East/West,35.21194,-80.85917,[East/West],[134.62]
